# Aula 09: Laboratório de Fixação e Simulado Integrado N1 (Redis + MongoDB)
### Tecnologia em Análise e Desenvolvimento de Sistemas (ADS) — IFCE Campus Tauá
**Componente:** Banco de Dados Não-Relacionais (ADS16)  
**Docente:** Prof. Me. Reginaldo Fernandes

---

## 🎯 Objetivos Deste Laboratório de Revisão
1. Revisar os fundamentos de arquitetura distribuída: Teorema CAP, Teorema PACELC e Propriedades ACID vs BASE.
2. Praticar o padrão **Cache-Aside** integrando **Redis** e **MongoDB** via Python.
3. Validar a **Regra de Ouro ESR** (*Equality, Sort, Range*) e a eliminação de *COLLSCANs* com `explain()`.
4. Executar pipelines analíticos no **MongoDB Aggregation Framework**.

In [ ]:
import redis
from pymongo import MongoClient
import json

# Inicializando clientes de conexao
r = redis.Redis(host='localhost', port=6379, decode_responses=True)
m = MongoClient('mongodb://localhost:27017')
db = m['ecommerce_shopgrids']

print("Redis Ping:", r.ping())
print("MongoDB Ping:", db.command('ping'))

## Desafio 1: Padrão Cache-Aside (Lazy Loading)
Implemente a consulta com fallback automático para o MongoDB e gravação com TTL de 60s.

In [ ]:
def obter_produto_cache(sku):
    chave = f"produto:{sku}"
    cache = r.get(chave)
    if cache:
        return json.loads(cache), "CACHE_HIT"
    
    doc = db.produtos.find_one({"sku": sku})
    if doc:
        doc["_id"] = str(doc["_id"])
        r.setex(chave, 60, json.dumps(doc))
        return doc, "CACHE_MISS"
    return None, "NAO_ENCONTRADO"

prod, status = obter_produto_cache("SKU-ELE-001")
print(f"Status: {status} | Produto: {prod['nome']} | Preço: R$ {prod['preco']}")

## Desafio 2: Pipeline de Agregação e Relatório de Vendas
Agregação dos pedidos por canal de venda com faturamento e ticket médio.

In [ ]:
pipeline = [
    {"$match": {"status": "entregue"}},
    {
        "$group": {
            "_id": "$canal",
            "faturamento": {"$sum": "$valor_total"},
            "pedidos": {"$sum": 1},
            "ticket_medio": {"$avg": "$valor_total"}
        }
    },
    {"$sort": {"faturamento": -1}}
]

list(db.pedidos.aggregate(pipeline))